# Pipeline: Qwen2.5-32B-Instruct

This notebook is set up for a Colab workflow: install dependencies, confirm the GPU, pull the repo, mount Drive, configure dataset and output paths, then run activation extraction and verify the saved HDF5 file.

## 1. Runtime Setup

Run the next two cells first. They install the Python dependencies used by the extraction pipeline and show the current GPU.

In [1]:
!pip install -q transformers accelerate h5py huggingface_hub scikit-learn

In [2]:
!nvidia-smi

Mon Apr 27 15:01:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             50W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Repo and Drive

The next cells pull the latest repo into `/content` and mount Google Drive. The code lives best on local Colab disk for speed, while datasets and final outputs can stay in Drive for persistence.

In [3]:
import os, sys

REPO_DIR = '/content/emotion-mechanisms-llm'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/daspushpita/emotion-mechanisms-llm.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

for p in [f'{REPO_DIR}/src', f'{REPO_DIR}/scripts']:
    if p not in sys.path:
        sys.path.insert(0, p)

Cloning into '/content/emotion-mechanisms-llm'...
remote: Enumerating objects: 203, done.
remote: Counting objects: 100% (203/203), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 203 (delta 107), reused 159 (delta 64), pack-reused 0 (from 0)
Receiving objects: 100% (203/203), 1.16 MiB | 17.23 MiB/s, done.
Resolving deltas: 100% (107/107), done.


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Configure Paths and Model

Edit `DATA_ROOT` if your Drive folder name differs. This is also the place to switch between smoke-test settings and a real 32B run. If you keep the sample values below, note that the current config still points to a 7B output filename and a 7B analysis model.

In [5]:
# Patch config BEFORE importing build_emotion_vectors so its module-level
# `from emotion_mechanisms.config import ...` picks up the Drive paths.
import emotion_mechanisms.config as cfg
from pathlib import Path

REPO_DIR = Path("/content/emotion-mechanisms-llm")
DATA_ROOT = Path("/content/drive/MyDrive/emotion-mechanisms-llm")

cfg.EMOTIONAL_STORIES_DATASET = DATA_ROOT / "datasets/processed/emotional_stories_qwen32B_v1_clean.jsonl"
cfg.NEUTRAL_STORIES_DATASET   = DATA_ROOT / "datasets/processed/neutral_stories_qwen32B_v1.jsonl"
cfg.ACTIVATIONS_PATH          = Path("/content/activations_32b.h5")
cfg.ANALYSIS_MODEL_32B        = "Qwen/Qwen2.5-32B-Instruct"
cfg.TOKEN_POSITION            = "mean"


assert cfg.EMOTIONAL_STORIES_DATASET.exists(), f'Missing: {cfg.EMOTIONAL_STORIES_DATASET}'
assert cfg.NEUTRAL_STORIES_DATASET.exists(),   f'Missing: {cfg.NEUTRAL_STORIES_DATASET}'
cfg.ACTIVATIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
print('Datasets found. Output ->', cfg.ACTIVATIONS_PATH)

Datasets found. Output -> /content/activations_32b.h5


In [7]:
from huggingface_hub import notebook_login
notebook_login()

## 4. Authenticate and Run

Hugging Face login is only needed if the model download requires authentication or if you want authenticated rate limits. The extraction cell below uses `max_stories=1` as a quick smoke test; increase it once the pipeline behaves the way you want.

In [8]:
import importlib
import build_emotion_vectors
importlib.reload(build_emotion_vectors)

build_emotion_vectors.main()

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Emotional stories:   0%|          | 0/2295 [00:00<?, ?it/s]

Neutral stories:   0%|          | 0/48 [00:00<?, ?it/s]

In [9]:
import shutil

drive_out = DATA_ROOT / "results/activations/activations_32b.h5"
drive_out.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2("/content/activations_32b.h5", drive_out)
print("Copied to", drive_out)

Copied to /content/drive/MyDrive/emotion-mechanisms-llm/results/activations/activations_32b.h5


In [10]:
import h5py
with h5py.File(cfg.ACTIVATIONS_PATH, 'r') as f:
    print('Top-level keys:', list(f.keys()))
    print('Emotions stored:', list(f['emotional'].keys()))
    first_layer = list(f['emotional/happy'].keys())[0]
    print(f'emotional/happy/{first_layer} shape:', f[f'emotional/happy/{first_layer}'].shape)

Top-level keys: ['emotional', 'neutral']
Emotions stored: ['afraid', 'angry', 'calm', 'desperate', 'guilty', 'happy', 'inspired', 'loving', 'nervous', 'proud', 'sad', 'surprised']
emotional/happy/layer_0 shape: (190, 5120)


## 5. Verify the Saved Activations

These checks confirm that the HDF5 file was written correctly. First inspect the top-level groups and one example tensor shape, then run a quick numeric sanity check across a few layers per emotion.

In [11]:
import numpy as np
import h5py

with h5py.File(cfg.ACTIVATIONS_PATH, 'r') as f:
    print(f"{'Emotion':<12} {'Layer':<10} {'Mean':>10} {'Std':>10} {'NaN':>6} {'Zero':>6}")
    print("-" * 56)
    for emotion in f['emotional'].keys():
        layers = list(f[f'emotional/{emotion}'].keys())
        # spot-check first, middle, last layer
        for layer in [layers[0], layers[len(layers)//2], layers[-1]]:
            arr = f[f'emotional/{emotion}/{layer}'][:]
            print(f"{emotion:<12} {layer:<10} {arr.mean():>10.3f} {arr.std():>10.3f} "
                f"{str(np.isnan(arr).any()):>6} {str((arr==0).all(-1).any()):>6}")
    
    # Also check neutral
    print("\nNeutral keys:", list(f['neutral'].keys()))
    n_layers = list(f['emotional/happy'].keys())
    print(f"Layers saved per emotion: {len(n_layers)} (e.g. {n_layers[0]} … {n_layers[-1]})")

Emotion      Layer            Mean        Std    NaN   Zero
--------------------------------------------------------
afraid       layer_0        -0.004      0.392  False  False
afraid       layer_42       -0.008      4.475  False  False
afraid       layer_56       -0.014     10.635  False  False
angry        layer_0        -0.004      0.392  False  False
angry        layer_42       -0.008      4.308  False  False
angry        layer_56       -0.022     10.372  False  False
calm         layer_0        -0.004      0.391  False  False
calm         layer_42       -0.000      4.493  False  False
calm         layer_56       -0.017     10.690  False  False
desperate    layer_0        -0.004      0.391  False  False
desperate    layer_42       -0.008      4.507  False  False
desperate    layer_56       -0.013     10.666  False  False
guilty       layer_0        -0.004      0.391  False  False
guilty       layer_42       -0.007      4.148  False  False
guilty       layer_56       -0.010     10.3